# HelpDesk AI: Chatbot using Sequence-to-Sequence (Seq2Seq) with Attention Mechanism

### Project Overview
This notebook demonstrates building **HelpDesk AI**, a generative chatbot using a **Sequence-to-Sequence (Seq2Seq)** neural network architecture with **Bahdanau Attention Mechanism** in **PyTorch**.

---
### Pipeline Structure
1. **Phase 1: Environment & Setup** - Dependency imports & device configuration.
2. **Phase 2: Data Preprocessing & Vocabulary** - Text cleaning, contraction expansion, and token-to-index mapping.
3. **Phase 3: Dataset & DataLoader** - PyTorch Dataset with dynamic batch padding.
4. **Phase 4: Deep Learning Architecture** - Encoder LSTM, Bahdanau Attention Layer, Decoder LSTM, and Seq2Seq wrapper.
5. **Phase 5: Training Loop** - Cross-Entropy Loss optimization with Teacher Forcing.
6. **Phase 6: Convergence Metrics** - Plotting Training Loss and Perplexity curves.
7. **Phase 7: Inference & Attention Heatmap** - Generating responses and plotting dynamic attention alignment matrices.

## Phase 1: Environment & Setup

In [ ]:
import os
import re
import json
import math
import time
import random
import unicodedata
from collections import Counter
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[+] Active Training Device: {device}')

## Phase 2: Data Preprocessing & Vocabulary Building

Dialogue text preprocessing includes:
- Normalizing unicode strings to plain ASCII.
- Expanding standard contractions (`don't` -> `do not`, `what's` -> `what is`).
- Cleaning out emojis and irregular symbols while preserving sentence-ending punctuation.
- Managing special reserved tokens:
  - `<PAD>` (Index 0): Padding shorter sentences in a batch.
  - `<SOS>` (Index 1): Start of Sentence signal for decoder.
  - `<EOS>` (Index 2): End of Sentence termination signal.
  - `<UNK>` (Index 3): Out-of-vocabulary unknown words.

In [ ]:
PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

def unicode_to_ascii(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def normalize_text(text: str) -> str:
    text = unicode_to_ascii(text.lower().strip())
    contractions = {
        r"can\'t": "cannot", r"won\'t": "will not", r"n\'t": " not",
        r"\'re": " are", r"\'s": " is", r"\'d": " would", r"\'ll": " will",
        r"\'ve": " have", r"\'m": " am", r"whats": "what is", r"dont": "do not",
        r"doesnt": "does not", r"didnt": "did not", r"hows": "how is"
    }
    for pat, rep in contractions.items():
        text = re.sub(pat, rep, text)
    text = re.sub(r"([.!?])", r" \1", text)
    text = re.sub(r"[^a-zA-Z0-9.!?]+", r" ", text)
    return re.sub(r"\s+", r" ", text).strip()

class Vocabulary:
    def __init__(self, name="helpdesk_vocab"):
        self.name = name
        self.word2index = {"<PAD>": PAD_IDX, "<SOS>": SOS_IDX, "<EOS>": EOS_IDX, "<UNK>": UNK_IDX}
        self.index2word = {idx: w for w, idx in self.word2index.items()}
        self.word2count = Counter()
        self.num_words = 4

    def add_sentence(self, sentence: str):
        for word in sentence.split():
            self.add_word(word)

    def add_word(self, word: str):
        self.word2count[word] += 1
        if word not in self.word2index:
            self.word2index[word] = self.num_words
            self.index2word[self.num_words] = word
            self.num_words += 1

    def sentence_to_indices(self, sentence: str, add_sos: bool = True, add_eos: bool = True) -> List[int]:
        tokens = []
        if add_sos:
            tokens.append(SOS_IDX)
        for word in sentence.split():
            tokens.append(self.word2index.get(word, UNK_IDX))
        if add_eos:
            tokens.append(EOS_IDX)
        return tokens

    def indices_to_sentence(self, indices: List[int], remove_special: bool = True) -> str:
        words = []
        for idx in indices:
            if idx == EOS_IDX and remove_special:
                break
            if remove_special and idx in (PAD_IDX, SOS_IDX):
                continue
            words.append(self.index2word.get(idx, "<UNK>"))
        return " ".join(words)

## Phase 3: Dataset Loading & Dynamic Collation

In [ ]:
class DialogueDataset(Dataset):
    def __init__(self, pairs: List[Tuple[str, str]], vocab: Vocabulary, max_len: int = 25):
        self.pairs = pairs
        self.vocab = vocab
        self.processed = []
        for q, a in pairs:
            src_idx = vocab.sentence_to_indices(q, add_sos=False, add_eos=True)[:max_len]
            trg_idx = vocab.sentence_to_indices(a, add_sos=True, add_eos=True)[:max_len]
            self.processed.append((torch.tensor(src_idx, dtype=torch.long), torch.tensor(trg_idx, dtype=torch.long)))

    def __len__(self):
        return len(self.processed)

    def __getitem__(self, idx):
        return self.processed[idx]

def pad_collate_fn(batch):
    src_seqs, trg_seqs = zip(*batch)
    src_lens = torch.tensor([len(s) for s in src_seqs], dtype=torch.long)
    trg_lens = torch.tensor([len(t) for t in trg_seqs], dtype=torch.long)
    src_pad = nn.utils.rnn.pad_sequence(src_seqs, batch_first=True, padding_value=PAD_IDX)
    trg_pad = nn.utils.rnn.pad_sequence(trg_seqs, batch_first=True, padding_value=PAD_IDX)
    return src_pad, trg_pad, src_lens, trg_lens

# Load dialogue corpus (auto-generates data/conversations.json if running in Google Colab / cloud)
data_path = os.path.join('data', 'conversations.json')
if not os.path.exists(data_path):
    os.makedirs('data', exist_ok=True)
    sample_data = [
  {
    "question": "hi",
    "answer": "hello ! how can i help you today ?"
  },
  {
    "question": "hello",
    "answer": "hello there ! how can i assist you today ?"
  },
  {
    "question": "hey there",
    "answer": "hey ! what can i do for you ?"
  },
  {
    "question": "good morning",
    "answer": "good morning ! how can i help you today ?"
  },
  {
    "question": "good afternoon",
    "answer": "good afternoon ! what brings you here today ?"
  },
  {
    "question": "good evening",
    "answer": "good evening ! how may i assist you ?"
  },
  {
    "question": "how are you",
    "answer": "i am doing great , thank you for asking ! how are you ?"
  },
  {
    "question": "how are you doing",
    "answer": "i am functioning smoothly and ready to assist you ."
  },
  {
    "question": "who are you",
    "answer": "i am helpdesk ai , your intelligent customer support assistant ."
  },
  {
    "question": "what is your name",
    "answer": "my name is helpdesk ai ."
  },
  {
    "question": "what can you do",
    "answer": "i can answer questions , troubleshoot issues , and assist with your account ."
  },
  {
    "question": "thank you",
    "answer": "you are welcome ! let me know if you need anything else ."
  },
  {
    "question": "thanks a lot",
    "answer": "glad i could help ! have a great day ."
  },
  {
    "question": "bye",
    "answer": "goodbye ! have a wonderful day ahead ."
  },
  {
    "question": "see you later",
    "answer": "see you ! feel free to return if you need more help ."
  },
  {
    "question": "i forgot my password",
    "answer": "you can reset your password by clicking forgot password on the login page ."
  },
  {
    "question": "how do i reset password",
    "answer": "click on forgot password to receive a reset link via email ."
  },
  {
    "question": "how to change my password",
    "answer": "go to account settings , select security , and click change password ."
  },
  {
    "question": "i cannot login to my account",
    "answer": "please verify your email and password or use password recovery ."
  },
  {
    "question": "my account is locked",
    "answer": "please wait fifteen minutes or contact support to unlock your account ."
  },
  {
    "question": "how do i create an account",
    "answer": "click sign up on our homepage and follow the registration steps ."
  },
  {
    "question": "where can i update my profile",
    "answer": "navigate to settings and click on edit profile details ."
  },
  {
    "question": "how do i delete my account",
    "answer": "you can request account deletion in the privacy settings menu ."
  },
  {
    "question": "how much does it cost",
    "answer": "we offer starter , pro , and enterprise plans on our pricing page ."
  },
  {
    "question": "what are the pricing plans",
    "answer": "our plans start at ten dollars per month with annual discounts ."
  },
  {
    "question": "is there a free trial",
    "answer": "yes , we offer a fourteen day free trial with full access ."
  },
  {
    "question": "how do i cancel my subscription",
    "answer": "you can cancel anytime from your billing and subscriptions dashboard ."
  },
  {
    "question": "can i get a refund",
    "answer": "refunds are available within thirty days of purchase upon request ."
  },
  {
    "question": "how do i update payment method",
    "answer": "go to billing settings and click update payment details ."
  },
  {
    "question": "what payment methods do you accept",
    "answer": "we accept major credit cards , paypal , and wire transfers ."
  },
  {
    "question": "where is my invoice",
    "answer": "invoices are downloadable from your billing history tab ."
  },
  {
    "question": "the application is not working",
    "answer": "please try refreshing your browser or clearing your cache ."
  },
  {
    "question": "i am getting an error code",
    "answer": "could you please provide the exact error message or code ?"
  },
  {
    "question": "page is loading slowly",
    "answer": "check your internet connection and verify our system status page ."
  },
  {
    "question": "the app crashed",
    "answer": "please restart the application and ensure you have the latest update ."
  },
  {
    "question": "how do i report a bug",
    "answer": "submit a bug report through the help center or feedback form ."
  },
  {
    "question": "is the server down",
    "answer": "all systems are operational according to our live status dashboard ."
  },
  {
    "question": "how do i contact support",
    "answer": "you can reach our support team via email at support at company dot com ."
  },
  {
    "question": "can i talk to a human",
    "answer": "i will gladly connect you to a live support agent shortly ."
  },
  {
    "question": "connect me to an agent",
    "answer": "transferring you to an available human specialist now ."
  },
  {
    "question": "what are your support hours",
    "answer": "our support team is available twenty four hours a day , seven days a week ."
  },
  {
    "question": "where is your office located",
    "answer": "our headquarters is located in san francisco , california ."
  },
  {
    "question": "do you have an api",
    "answer": "yes , our restful api documentation is available at docs dot company dot com ."
  },
  {
    "question": "where can i find documentation",
    "answer": "complete guides and api references are available in our developer portal ."
  },
  {
    "question": "how do i integrate webhooks",
    "answer": "you can configure webhook endpoints in the developer settings section ."
  },
  {
    "question": "how do i generate an api key",
    "answer": "go to developer settings and click generate new api key ."
  },
  {
    "question": "where is my order",
    "answer": "you can track your order status using your tracking number ."
  },
  {
    "question": "how do i track shipment",
    "answer": "enter your order id in the tracking portal to view real time updates ."
  },
  {
    "question": "how long does delivery take",
    "answer": "standard delivery takes three to five business days ."
  },
  {
    "question": "what is the return policy",
    "answer": "items can be returned within thirty days in original condition ."
  },
  {
    "question": "what is machine learning",
    "answer": "machine learning is a subset of ai enabling systems to learn from data ."
  },
  {
    "question": "what is natural language processing",
    "answer": "nlp is the branch of ai helping computers understand and generate human language ."
  },
  {
    "question": "how does attention mechanism work",
    "answer": "attention allows the decoder to focus on relevant parts of the input sentence ."
  },
  {
    "question": "what is seq2seq",
    "answer": "seq2seq is an encoder decoder architecture that transforms one sequence into another ."
  },
  {
    "question": "why use lstm instead of rnn",
    "answer": "lstms prevent the vanishing gradient problem and retain long term context ."
  },
  {
    "question": "tell me a joke",
    "answer": "why do programmers prefer dark mode ? because light attracts bugs !"
  },
  {
    "question": "what is the weather today",
    "answer": "i do not have live weather sensors , but i hope it is sunny outside !"
  },
  {
    "question": "are you real",
    "answer": "i am an artificial intelligence designed to help you with your questions ."
  },
  {
    "question": "what is your purpose",
    "answer": "my purpose is to provide fast , helpful , and accurate answers ."
  },
  {
    "question": "can you help me",
    "answer": "of course ! please let me know what issue you are facing ."
  },
  {
    "question": "i have a question",
    "answer": "feel free to ask anything , and i will do my best to assist you ."
  },
  {
    "question": "nice to meet you",
    "answer": "it is a pleasure to meet you too ! how can i assist you today ?"
  },
  {
    "question": "have a nice day",
    "answer": "thank you ! wishing you a wonderful day as well ."
  }
]
    with open(data_path, 'w', encoding='utf-8') as f:
        json.dump(sample_data, f, indent=2)
    print('[+] Auto-created data/conversations.json for cloud environment.')
with open(data_path, 'r', encoding='utf-8') as f:
    raw_pairs = json.load(f)

pairs = [(normalize_text(item['question']), normalize_text(item['answer'])) for item in raw_pairs]
vocab = Vocabulary()
for q, a in pairs:
    vocab.add_sentence(q)
    vocab.add_sentence(a)

dataset = DialogueDataset(pairs, vocab)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=pad_collate_fn)
print(f'[+] Loaded {len(pairs)} dialogue pairs. Total Vocabulary Size: {vocab.num_words} tokens.')

## Phase 4: Seq2Seq Architecture with Bahdanau Attention

### Architecture Components:
1. **Encoder**: Processes variable-length input tokens with a **Bidirectional LSTM**, capturing forward and backward contextual relationships.
2. **Bahdanau Attention Layer**: Calculates alignment scores between the previous decoder state $s_{i-1}$ and all encoder states $h_j$:
   $$e_{ij} = v_a^T \tanh(W_a s_{i-1} + U_a h_j)$$
   $$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k} \exp(e_{ik})}, \quad c_i = \sum_j \alpha_{ij} h_j$$
3. **Decoder**: Generates tokens conditioned on both the previous token embedding and the dynamic context vector $c_i$.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim: int, emb_dim: int, enc_hid_dim: int, dec_hid_dim: int, n_layers: int = 1, dropout: float = 0.2):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=PAD_IDX)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.LSTM(emb_dim, enc_hid_dim, num_layers=n_layers, bidirectional=True, batch_first=True)
        self.fc_hidden = nn.Linear(enc_hid_dim * 2, dec_hid_dim)
        self.fc_cell = nn.Linear(enc_hid_dim * 2, dec_hid_dim)

    def forward(self, src: torch.Tensor, src_lengths: Optional[torch.Tensor] = None):
        embedded = self.dropout(self.embedding(src))
        if src_lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(embedded, src_lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, (hidden, cell) = self.rnn(packed)
            outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            outputs, (hidden, cell) = self.rnn(embedded)
            
        hidden_cat = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
        cell_cat = torch.cat((cell[-2, :, :], cell[-1, :, :]), dim=1)
        dec_init_hidden = torch.tanh(self.fc_hidden(hidden_cat)).unsqueeze(0)
        dec_init_cell = torch.tanh(self.fc_cell(cell_cat)).unsqueeze(0)
        return outputs, (dec_init_hidden, dec_init_cell)


class Attention(nn.Module):
    def __init__(self, enc_hid_dim: int, dec_hid_dim: int):
        super().__init__()
        self.W_dec = nn.Linear(dec_hid_dim, dec_hid_dim, bias=False)
        self.W_enc = nn.Linear(enc_hid_dim * 2, dec_hid_dim, bias=False)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, dec_hidden: torch.Tensor, enc_outputs: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        src_len = enc_outputs.shape[1]
        dec_rep = dec_hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.W_dec(dec_rep) + self.W_enc(enc_outputs))
        scores = self.v(energy).squeeze(2)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e10)
        return F.softmax(scores, dim=1)


class Decoder(nn.Module):
    def __init__(self, output_dim: int, emb_dim: int, enc_hid_dim: int, dec_hid_dim: int, attention: Attention, dropout: float = 0.2):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_IDX)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.LSTM(emb_dim + enc_hid_dim * 2, dec_hid_dim, batch_first=True)
        self.fc_out = nn.Linear(dec_hid_dim + enc_hid_dim * 2 + emb_dim, output_dim)

    def forward(self, input_token: torch.Tensor, hidden_state: Tuple[torch.Tensor, torch.Tensor], enc_outputs: torch.Tensor, mask: Optional[torch.Tensor] = None):
        hidden, cell = hidden_state
        input_token = input_token.unsqueeze(1)
        embedded = self.dropout(self.embedding(input_token))
        
        a = self.attention(hidden[-1], enc_outputs, mask)
        context = torch.bmm(a.unsqueeze(1), enc_outputs)
        
        rnn_input = torch.cat((embedded, context), dim=2)
        rnn_output, (new_hidden, new_cell) = self.rnn(rnn_input, (hidden, cell))
        
        prediction = self.fc_out(torch.cat((rnn_output.squeeze(1), context.squeeze(1), embedded.squeeze(1)), dim=1))
        return prediction, (new_hidden, new_cell), a


class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder, device: torch.device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src: torch.Tensor, trg: torch.Tensor, src_lengths: Optional[torch.Tensor] = None, teacher_forcing_ratio: float = 0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        outputs = torch.zeros(batch_size, trg_len, self.decoder.output_dim).to(self.device)
        enc_outputs, hidden_state = self.encoder(src, src_lengths)
        mask = (src != PAD_IDX).to(self.device)
        
        input_token = trg[:, 0]
        for t in range(1, trg_len):
            output, hidden_state, _ = self.decoder(input_token, hidden_state, enc_outputs, mask)
            outputs[:, t, :] = output
            teacher_force = random.random() < teacher_forcing_ratio
            input_token = trg[:, t] if teacher_force else output.argmax(1)
        return outputs

## Phase 5: Training Loop with Teacher Forcing

We train the model using **Teacher Forcing** (probability $p=0.6$), **Cross-Entropy Loss** with PAD token masking, and **Adam Optimizer**.

In [ ]:
EMB_DIM = 128
ENC_HID_DIM = 256
DEC_HID_DIM = 256

encoder = Encoder(vocab.num_words, EMB_DIM, ENC_HID_DIM, DEC_HID_DIM)
attention = Attention(ENC_HID_DIM, DEC_HID_DIM)
decoder = Decoder(vocab.num_words, EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, attention)
model = Seq2SeqAttention(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

epochs = 60
history = {'train_loss': [], 'train_ppl': []}

print('[*] Starting Training...')
start_time = time.time()

for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0
    for src, trg, src_lens, _ in dataloader:
        src, trg, src_lens = src.to(device), trg.to(device), src_lens.to(device)
        optimizer.zero_grad()
        output = model(src, trg, src_lens, teacher_forcing_ratio=0.6)
        
        output_dim = output.shape[-1]
        output = output[:, 1:].contiguous().view(-1, output_dim)
        trg = trg[:, 1:].contiguous().view(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(dataloader)
    ppl = math.exp(min(avg_loss, 100))
    history['train_loss'].append(avg_loss)
    history['train_ppl'].append(ppl)
    
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch: {epoch:02d}/{epochs:02d} | Cross-Entropy Loss: {avg_loss:.4f} | Perplexity: {ppl:.2f}')

total_time = time.time() - start_time
print(f'[+] Training Finished in {total_time:.1f}s!')

## Phase 6: Plot Loss & Perplexity Convergence Curves

In [ ]:
%matplotlib inline
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
epochs_range = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_range, history['train_loss'], 'o-', color='#2563eb', linewidth=2)
ax1.set_title('Cross-Entropy Training Loss', fontweight='bold', fontsize=12)
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('Loss', fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.plot(epochs_range, history['train_ppl'], 's-', color='#059669', linewidth=2)
ax2.set_title('Perplexity (PPL = exp(Loss))', fontweight='bold', fontsize=12)
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('Perplexity', fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Phase 7: Interactive Chat & Attention Heatmap Alignment

In [ ]:
def chat_and_visualize(model: Seq2SeqAttention, vocab: Vocabulary, user_query: str, device: torch.device):
    model.eval()
    cleaned = normalize_text(user_query)
    src_indices = vocab.sentence_to_indices(cleaned, add_sos=False, add_eos=True)
    src_tensor = torch.tensor(src_indices, dtype=torch.long).unsqueeze(0).to(device)
    src_len = torch.tensor([len(src_indices)], dtype=torch.long)
    
    with torch.no_grad():
        enc_out, hidden_state = model.encoder(src_tensor, src_len)
        mask = (src_tensor != PAD_IDX).to(device)
        input_token = torch.tensor([SOS_IDX], dtype=torch.long).to(device)
        
        out_indices = []
        attn_matrix = []
        for _ in range(25):
            out, hidden_state, attn = model.decoder(input_token, hidden_state, enc_out, mask)
            attn_matrix.append(attn.squeeze(0).cpu().numpy())
            top1 = out.argmax(1).item()
            if top1 == EOS_IDX:
                break
            out_indices.append(top1)
            input_token = torch.tensor([top1], dtype=torch.long).to(device)
            
    response = vocab.indices_to_sentence(out_indices)
    in_tokens = cleaned.split() + ['<EOS>']
    out_tokens = [vocab.index2word.get(idx, '<UNK>') for idx in out_indices]
    
    print(f'User : "{user_query}"')
    print(f'Bot  : "{response.capitalize()}"')
    
    # Plot attention heatmap alignment
    if len(attn_matrix) > 0:
        fig, ax = plt.subplots(figsize=(max(6, len(in_tokens) * 0.8), max(4, len(out_tokens) * 0.5)))
        cax = ax.matshow(np.array(attn_matrix), cmap='YlGnBu')
        fig.colorbar(cax, shrink=0.8)
        ax.set_xticks(range(len(in_tokens)))
        ax.set_yticks(range(len(out_tokens)))
        ax.set_xticklabels(in_tokens, rotation=45, ha='left', fontsize=10)
        ax.set_yticklabels(out_tokens, fontsize=10)
        ax.set_xlabel('Encoder Input Tokens', fontweight='bold', labelpad=10)
        ax.set_ylabel('Decoder Output Tokens', fontweight='bold', labelpad=10)
        plt.title('Bahdanau Attention Alignment Matrix', pad=25, fontweight='bold')
        plt.tight_layout()
        plt.show()

# Test chatbot on sample queries
chat_and_visualize(model, vocab, 'How do I reset password', device)
chat_and_visualize(model, vocab, 'Who are you', device)
chat_and_visualize(model, vocab, 'How much does it cost', device)